# 🧠 Week 1: First Principles & Hardware Bound Physics
## The Roofline Model, Arithmetic Intensity & Memory-Bound Autoregressive Decode

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/simranjeet97/AI_Engineer_Roadmap_2027/blob/main/notebooks/week_01_roofline_and_hardware_bounds.ipynb)
[![Curriculum](https://img.shields.io/badge/Curriculum-10%20Weeks%20%7C%2050%20Sessions-blue.svg)](https://github.com/simranjeet97/AI_Engineer_Roadmap_2027)
[![Author](https://img.shields.io/badge/Author-Simranjeet%20Singh-black.svg)](https://github.com/simranjeet97)

---

### Staff Engineer Operational Thesis
> *"If you do not understand the physics of High-Bandwidth Memory (HBM) and the arithmetic intensity of linear algebra kernels, you will spend months writing software optimizations that run headfirst into brick walls built by hardware limits."*

In modern frontier AI research labs (OpenAI, Anthropic, DeepSeek, Meta), inference engineering is governed by one immutable rule: **Hardware Physics Always Wins**. 

Most software developers believe that adding more GPUs or writing custom prompt templates speeds up generation. In reality, during the autoregressive token generation (decode) phase, modern GPUs operate at **less than 1% to 3% of their theoretical peak Tensor Core compute capacity**. They are almost completely starved for memory bandwidth.

### 🎯 Week 1 Learning Objectives:
1. **The Roofline Model**: Calculate operational intensity ($I = 	ext{FLOPs} / 	ext{Byte}$) and derive the exact "knee point" where compute transitions to memory starvation.
2. **Prefill GEMM vs Decode GEMV**: Prove mathematically why prompt prefill is compute-bound ($O(N^2)$), whereas token decode is memory-bound ($O(N)$).
3. **The KV Cache Memory Formula**: Compute exact memory footprints across MHA (Multi-Head), MQA (Multi-Query), and GQA (Grouped-Query Attention) for models up to 70B parameters.
4. **Empirical CUDA Profiling**: Use `torch.cuda.Event` to profile GPU memory bus utilization and prove that decode GEMV saturates memory bandwidth while Tensor Cores sit idle.
5. **The Hardware Constraints Envelope**: Construct a high-resolution interactive Roofline chart plotting operational points for batch sizes $B=1 \dots 256$.


---
## 🛠️ Step 0: Environment Detection & Hardware Accelerator Database

We query your active Colab execution environment (CPU, T4, L4, V100, or A100) and initialize a reference database of industry-standard AI inference accelerators (NVIDIA H100 SXM5, A100 80GB, L40S, L4, T4). This allows us to compare your active hardware against frontier datacenter silicon.


In [ ]:
# @title Step 0: Accelerator Detection & Hardware Reference Database
import os
import sys
import math
import time
from typing import Dict, Any, Tuple

try:
    import torch
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
except ImportError:
    !pip install -q torch numpy pandas matplotlib seaborn
    import torch
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns

# Configure visual styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11

# Comprehensive Datacenter & Edge Accelerator Database
ACCELERATOR_DB: Dict[str, Dict[str, Any]] = {
    "NVIDIA H100 SXM5 (80GB)": {
        "peak_fp16_tflops": 989.0,   # Dense Tensor Core FP16/BF16
        "peak_bandwidth_gb_s": 3350.0, # HBM3
        "vram_gb": 80.0,
        "interconnect": "NVLink 4 (900 GB/s)"
    },
    "NVIDIA H100 PCIe (80GB)": {
        "peak_fp16_tflops": 756.0,
        "peak_bandwidth_gb_s": 2000.0, # HBM2e
        "vram_gb": 80.0,
        "interconnect": "PCIe Gen5 (128 GB/s)"
    },
    "NVIDIA A100 SXM4 (80GB)": {
        "peak_fp16_tflops": 312.0,   # Dense Tensor Core FP16/BF16
        "peak_bandwidth_gb_s": 2039.0, # HBM2e
        "vram_gb": 80.0,
        "interconnect": "NVLink 3 (600 GB/s)"
    },
    "NVIDIA L40S (48GB)": {
        "peak_fp16_tflops": 362.0,   # Ada Lovelace FP16 Tensor
        "peak_bandwidth_gb_s": 864.0,  # GDDR6
        "vram_gb": 48.0,
        "interconnect": "PCIe Gen4 (64 GB/s)"
    },
    "NVIDIA L4 (24GB)": {
        "peak_fp16_tflops": 120.0,   # Ada Lovelace
        "peak_bandwidth_gb_s": 300.0,  # GDDR6
        "vram_gb": 24.0,
        "interconnect": "PCIe Gen4"
    },
    "NVIDIA T4 (16GB - Colab Default)": {
        "peak_fp16_tflops": 65.0,    # Turing Tensor Core
        "peak_bandwidth_gb_s": 320.0,  # GDDR6
        "vram_gb": 16.0,
        "interconnect": "PCIe Gen3"
    },
    "Apple M3 Max (128GB Unified)": {
        "peak_fp16_tflops": 36.0,
        "peak_bandwidth_gb_s": 400.0,
        "vram_gb": 128.0,
        "interconnect": "Unified Memory Bus"
    }
}

# Inspect Active Runtime
active_gpu = "None (CPU Mode)"
gpu_available = torch.cuda.is_available()

if gpu_available:
    device_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    active_gpu = f"{device_name} ({total_mem:.1f} GB)"
    print(f"✅ GPU Detected: {active_gpu}")
    print(f"   CUDA Capability: {torch.cuda.get_device_capability(0)}")
    print(f"   PyTorch Version: {torch.__version__}")
else:
    print("⚠️ No GPU detected! Running in CPU emulation mode.")
    print("   Tip: In Google Colab, go to Runtime -> Change runtime type -> Select T4, L4, or A100 GPU.")

# Display Accelerator Table
df_accelerators = pd.DataFrame.from_dict(ACCELERATOR_DB, orient='index')
df_accelerators["Knee Point (FLOPs/Byte)"] = (
    df_accelerators["peak_fp16_tflops"] * 1000.0 / df_accelerators["peak_bandwidth_gb_s"]
).round(2)
display(df_accelerators[["peak_fp16_tflops", "peak_bandwidth_gb_s", "vram_gb", "Knee Point (FLOPs/Byte)"]])


### 🔍 Code Explanation: Step 0
1. **`ACCELERATOR_DB`**: Maps theoretical hardware capabilities. Note the massive difference between compute capability (FLOPs) and memory bandwidth (GB/s).
2. **Knee Point**: Calculated as $rac{	ext{Peak FLOPs}}{	ext{Peak Bandwidth}}$. Notice that an **NVIDIA H100 SXM5 requires an operational intensity of $pprox 295 	ext{ FLOPs/Byte}$** before its Tensor Cores are fully utilized!
3. **`torch.cuda` Inspection**: Gathers physical hardware parameters from your active runtime.


---
## 📐 Day 1: The Roofline Equation & Operational Intensity

### The Mathematical Model
The Roofline Model establishes the upper bound on attainable computational performance as a function of the algorithm's **Operational Intensity** (or Arithmetic Intensity, denoted as $I$).

$$\text{Attainable Performance (TFLOPS)} = \min\left(\text{Peak Compute (TFLOPS)}, \; I \times \text{Memory Bandwidth (TB/s)}\right)$$

Where Operational Intensity $I$ is defined as:

$$I = \frac{\text{Total Arithmetic Operations (FLOPs)}}{\text{Total Memory Traffic (Bytes Transferred Across Memory Bus)}}$$

### Derivation of the "Knee Point"
The transition boundary between the **Memory-Bound Regime** and the **Compute-Bound Regime** occurs at the inflection point ("knee"):

$$I_{\text{knee}} = \frac{\text{Peak Compute Ceiling (TFLOPS)}}{\text{Peak Memory Bandwidth (TB/s)}} = \frac{\text{FLOPs}}{\text{Byte}}$$

* **If $I < I_{\text{knee}}$**: The algorithm is **Memory-Bound**. The execution time is strictly determined by how fast data can be read from High-Bandwidth Memory (HBM). Adding more compute cores yields **zero** speedup.
* **If $I \ge I_{\text{knee}}$**: The algorithm is **Compute-Bound**. The memory bus can deliver operands fast enough to keep the execution units (ALUs/Tensor Cores) fully saturated.

Let's write a simulator calculating the exact operational intensity of a linear projection layer ($Y = XW$) as batch size $B$ scales from $1$ to $2048$.


In [ ]:
# @title Day 1: Operational Intensity Simulation of a Transformer Linear Layer
def analyze_linear_layer_intensity(
    d_in: int = 4096,
    d_out: int = 4096,
    precision_bytes: int = 2,  # 2 bytes for FP16/BF16, 1 byte for FP8, 0.5 for INT4
    batch_sizes: list = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048]
) -> pd.DataFrame:
    """
    Computes Arithmetic Operations (FLOPs), Memory Traffic (Bytes), and Operational
    Intensity (FLOPs/Byte) for a linear layer Y = X @ W.
    
    Dimensions:
      X: [Batch Size (B), d_in]
      W: [d_in, d_out]
      Y: [Batch Size (B), d_out]
    """
    results = []
    
    # Weight matrix is static in HBM and must be loaded once per forward pass
    weight_elements = d_in * d_out
    weight_bytes = weight_elements * precision_bytes
    
    for b in batch_sizes:
        # 1. Total Arithmetic Operations (Multiply-Accumulate: 1 Multiply + 1 Add = 2 FLOPs per element)
        flops = 2 * b * d_in * d_out
        
        # 2. Total Memory Traffic (Bytes transferred from/to HBM)
        # We must read: Weights (W) + Input Activations (X)
        # We must write: Output Activations (Y)
        input_bytes = b * d_in * precision_bytes
        output_bytes = b * d_out * precision_bytes
        total_memory_traffic_bytes = weight_bytes + input_bytes + output_bytes
        
        # 3. Operational Intensity (I)
        intensity = flops / total_memory_traffic_bytes
        
        results.append({
            "Batch Size (B)": b,
            "Total GFLOPs": flops / 1e9,
            "Memory Traffic (MB)": total_memory_traffic_bytes / (1024**2),
            "Operational Intensity (FLOPs/Byte)": round(intensity, 3),
            "% of H100 Knee (295 FLOPs/B)": f"{(intensity / 295.2 * 100):.2f}%"
        })
        
    df = pd.DataFrame(results)
    return df

# Run the simulation for a standard 4096 -> 4096 projection layer (e.g., Llama-3-8B hidden dim)
df_intensity = analyze_linear_layer_intensity()
display(df_intensity)


### 🔍 Deep Dive: What Did the Day 1 Code Prove?

Look closely at the table above for **Batch Size $B = 1$**:
* Total Weight Bytes: $\approx 33.55 \text{ MB}$.
* Operational Intensity: **$0.999 \approx 1.000 \text{ FLOPs/Byte}$**!
* An NVIDIA H100 SXM5 requires **$295.2 \text{ FLOPs/Byte}$** to saturate its Tensor Cores.
* At $B = 1$, the operational intensity is **$0.34\%$ of the knee point**! 

> **Staff Engineer Insight**: When serving a single user ($B=1$), the GPU spends $99.66\%$ of its time waiting for weights to travel over the memory bus from HBM3 to SRAM registers. The Tensor Cores are practically dormant. Only when batch size reaches $B \ge 512$ does operational intensity approach the knee point.


---
## ⚡ Day 2: Anatomy of Attention: Prefill GEMM vs Autoregressive Decode GEMV

### The Fundamental Serving Asymmetry

Every LLM generation request consists of two fundamentally distinct computational regimes:

```
+----------------------------------------------------------------------------------------------------+
|  PHASE 1: PROMPT PREFILL                                                                           |
|  - Prompt length S is processed in one shot.                                                       |
|  - Matrix-Matrix Multiplication (GEMM): X [S, D] @ W [D, D]                                       |
|  - Operational Intensity is high (Tokens reuse the loaded weights simultaneously).                |
|  - REGIME: COMPUTE-BOUND (Tensor Cores saturated).                                                |
+----------------------------------------------------------------------------------------------------+
                                                |
                                                v
+----------------------------------------------------------------------------------------------------+
|  PHASE 2: AUTOREGRESSIVE DECODE                                                                    |
|  - Generates 1 token per sequence per forward step.                                                |
|  - Matrix-Vector Multiplication (GEMV): X [1, D] @ W [D, D]                                       |
|  - Entire model weights must be loaded from HBM to registers to compute JUST ONE TOKEN!            |
|  - REGIME: MEMORY-BOUND (Memory Bus saturated, Tensor Cores idle).                                 |
+----------------------------------------------------------------------------------------------------+
```

### The Memory Bandwidth Bottleneck Formula for Token Decode
For batch size $B=1$, each output token requires loading **all model parameters** once:

$$t_{\text{token}} \ge \frac{\text{Model Weight Size in Bytes}}{\text{Peak Memory Bandwidth (Bytes/s)}}$$

$$\text{Max Theoretical Decode Speed (tokens/sec)} = \frac{1}{t_{\text{token}}} = \frac{\text{Memory Bandwidth (Bytes/s)}}{\text{Model Weight Size in Bytes}}$$

Let's write a script calculating the maximum theoretical decode speed across different models (8B to 70B) and precisions (FP16, FP8, INT4).


In [ ]:
# @title Day 2: Single-Stream Decode Latency & Throughput Bounds Simulator
def simulate_model_decode_limits():
    """
    Calculates the hardware-imposed theoretical ceiling on single-stream token generation.
    """
    models = [
        {"name": "Llama-3-8B (FP16)", "params_b": 8.0, "bytes_per_param": 2.0},
        {"name": "Llama-3-8B (FP8)",  "params_b": 8.0, "bytes_per_param": 1.0},
        {"name": "Llama-3-8B (INT4)", "params_b": 8.0, "bytes_per_param": 0.5},
        {"name": "Llama-3-70B (FP16)","params_b": 70.0, "bytes_per_param": 2.0},
        {"name": "Llama-3-70B (FP8)", "params_b": 70.0, "bytes_per_param": 1.0},
        {"name": "Llama-3-70B (INT4)","params_b": 70.0, "bytes_per_param": 0.5},
    ]
    
    records = []
    
    for m in models:
        weight_gb = (m["params_b"] * 1e9 * m["bytes_per_param"]) / (1024**3)
        
        for gpu_name, gpu_spec in ACCELERATOR_DB.items():
            if "Unified" in gpu_name:
                continue # Skip edge for comparison
            bw_gb_s = gpu_spec["peak_bandwidth_gb_s"]
            
            # Theoretical minimum time to stream weights once (in milliseconds)
            min_time_per_token_ms = (weight_gb / bw_gb_s) * 1000.0
            
            # Theoretical maximum tokens/second for batch size = 1
            max_tokens_sec = 1000.0 / min_time_per_token_ms if min_time_per_token_ms > 0 else 0
            
            records.append({
                "Model": m["name"],
                "Weight Size (GB)": round(weight_gb, 1),
                "Accelerator": gpu_name,
                "Memory Bandwidth": f"{bw_gb_s:.0f} GB/s",
                "Min Latency / Token (ms)": round(min_time_per_token_ms, 2),
                "Max Theoretical tok/s (B=1)": round(max_tokens_sec, 1)
            })
            
    return pd.DataFrame(records)

df_decode_limits = simulate_model_decode_limits()

# Pivot table for clean presentation: Model vs Accelerator Max tok/s
pivot_tok_s = df_decode_limits.pivot(index="Model", columns="Accelerator", values="Max Theoretical tok/s (B=1)")
print("=== MAXIMUM THEORETICAL DECODE TOKENS/SEC (BATCH SIZE = 1) ===")
display(pivot_tok_s)

pivot_latency = df_decode_limits.pivot(index="Model", columns="Accelerator", values="Min Latency / Token (ms)")
print("
=== MINIMUM THEORETICAL TIME PER TOKEN (MS) ===")
display(pivot_latency)


### 🔍 Deep Dive: What Did the Day 2 Code Prove?

1. **Llama-3-70B in FP16 ($130.4 \text{ GB}$ weights)**:
   - On an **NVIDIA A100 80GB** ($2039 \text{ GB/s}$): Even if you could fit the weights across NVLinked GPUs, transferring 130GB takes **$64 \text{ ms}$ minimum** per token. Maximum possible speed: **$15.6 \text{ tok/s}$**!
   - On an **NVIDIA H100 SXM5** ($3350 \text{ GB/s}$): Minimum time per token is **$38.9 \text{ ms}$**. Maximum possible speed: **$25.7 \text{ tok/s}$**.
2. **The Power of Quantization**:
   - Compressing Llama-3-70B from FP16 ($130 \text{ GB}$) to INT4 ($32.6 \text{ GB}$) cuts the memory bandwidth demand by $4\times$. On an H100, theoretical throughput jumps from **$25.7 \text{ tok/s}$** to **$102.8 \text{ tok/s}$**!
   - This proves why quantization in LLM serving is primarily a **bandwidth accelerator**, not just a storage-saving technique!


---
## 💾 Day 3: The KV Cache Memory Formula (MHA vs MQA vs GQA)

### Why the KV Cache Exists
In autoregressive generation, each token attend to all previous tokens. To avoid recalculating the Key and Value vectors of past tokens at every step ($O(N^2)$ recomputation), we cache them in GPU VRAM.

### The Exact KV Cache Size Equation
For a transformer model, the memory consumed by the KV Cache across the cluster is:

$$\text{Size}_{\text{KV}} = 2 \times n_{\text{layers}} \times n_{\text{kv\_heads}} \times d_{\text{head}} \times \text{seq\_len} \times \text{bytes\_per\_elem} \times \text{batch\_size}$$

* **Factor $2$**: One tensor for Keys, one tensor for Values.
* **$n_{\text{layers}}$**: Number of transformer layers (e.g., 32 for 8B, 80 for 70B).
* **$n_{\text{kv\_heads}}$**: Number of Key/Value heads.
  * **MHA (Multi-Head Attention)**: $n_{\text{kv\_heads}} = n_{\text{q\_heads}}$ (e.g., Llama-2: 32 heads).
  * **MQA (Multi-Query Attention)**: $n_{\text{kv\_heads}} = 1$. All query heads share 1 KV head!
  * **GQA (Grouped-Query Attention)**: $n_{\text{kv\_heads}} = \frac{n_{\text{q\_heads}}}{G}$ (e.g., Llama-3: 8 KV heads vs 32 Q heads $\implies 4\times$ reduction!).
* **$d_{\text{head}}$**: Dimension of each attention head ($d_{\text{model}} / n_{\text{q\_heads}}$, typically 128).
* **$\text{bytes\_per\_elem}$**: 2 for FP16/BF16, 1 for FP8.

Let's build a calculator to determine when KV cache memory surpasses model weights memory and causes GPU OOM.


In [ ]:
# @title Day 3: KV Cache Memory Footprint & Crossover Calculator
def calculate_kv_cache_gb(
    n_layers: int,
    n_kv_heads: int,
    d_head: int,
    seq_len: int,
    batch_size: int,
    precision_bytes: int = 2
) -> float:
    """
    Computes exact KV cache size in Gigabytes (GiB).
    """
    total_bytes = 2 * n_layers * n_kv_heads * d_head * seq_len * precision_bytes * batch_size
    return total_bytes / (1024**3)

# Architectural Specs
MODELS_ARCH = {
    "Llama-2-7B (MHA: 32 KV Heads)": {
        "n_layers": 32, "n_q_heads": 32, "n_kv_heads": 32, "d_head": 128, "weights_gb": 14.0
    },
    "Llama-3-8B (GQA: 8 KV Heads)": {
        "n_layers": 32, "n_q_heads": 32, "n_kv_heads": 8, "d_head": 128, "weights_gb": 16.0
    },
    "Llama-3-70B (GQA: 8 KV Heads)": {
        "n_layers": 80, "n_q_heads": 64, "n_kv_heads": 8, "d_head": 128, "weights_gb": 140.0
    }
}

context_lengths = [2048, 8192, 32768, 65536, 131072]
batch_sizes = [1, 16, 64, 128]

records = []
for model_name, spec in MODELS_ARCH.items():
    for b in batch_sizes:
        for s in context_lengths:
            kv_gb = calculate_kv_cache_gb(
                spec["n_layers"], spec["n_kv_heads"], spec["d_head"], s, b, precision_bytes=2
            )
            records.append({
                "Model": model_name,
                "Batch Size": b,
                "Context (Tokens)": f"{s // 1024}k",
                "KV Cache (GB)": round(kv_gb, 2),
                "KV vs Model Weights": f"{(kv_gb / spec['weights_gb'] * 100):.1f}%"
            })

df_kv = pd.DataFrame(records)

# Filter view for Llama-3-8B vs Llama-2-7B at Batch Size 16
print("=== KV CACHE COMPARISON AT BATCH SIZE = 16 ===")
display(df_kv[df_kv["Batch Size"] == 16].pivot(index="Context (Tokens)", columns="Model", values="KV Cache (GB)"))

print("
=== KV CACHE MEMORY AS % OF MODEL WEIGHTS (LLAMA-3-8B, BATCH SIZE = 64) ===")
display(df_kv[(df_kv["Model"].str.contains("Llama-3-8B")) & (df_kv["Batch Size"] == 64)][["Context (Tokens)", "KV Cache (GB)", "KV vs Model Weights"]])


### 🔍 Deep Dive: What Did the Day 3 Code Prove?

1. **MHA vs GQA Impact**:
   - At $16\text{k}$ context and batch size $16$, Llama-2-7B (MHA) consumes **$32.0 \text{ GB}$** of KV cache—more than double its entire model weight ($14\text{ GB}$)!
   - Llama-3-8B (GQA with 8 heads) consumes only **$8.0 \text{ GB}$** for the identical sequence length—a **$4\times$ memory reduction**.
2. **The 128k Context Wall**:
   - For Llama-3-8B at batch size $64$ and $128\text{k}$ context, the KV cache alone requires **$256 \text{ GB}$ of VRAM**!
   - This proves why serving long contexts with high concurrency is impossible without **PagedAttention** (Week 2), **Prefix Caching** (Week 5), and **KV Cache Eviction** (Week 7).


---
## ⏱️ Day 4: Empirical CUDA Benchmarking: GEMV vs GEMM

### The Pitfall of Benchmarking GPUs in Python
Naively calling `time.time()` or `time.perf_counter()` to benchmark PyTorch CUDA operations produces completely invalid numbers. 
* CUDA kernel launches are **asynchronous**. The CPU returns control to Python immediately after scheduling the kernel on the CUDA driver queue, before the GPU has even started computing!
* To benchmark real GPU memory traffic and execution duration, we must use `torch.cuda.Event(enable_timing=True)` with explicit `cudaDeviceSynchronize()`.

Let's execute an empirical benchmark comparing:
1. **GEMV (Decode Simulation)**: Matrix-Vector multiplication $[1, 4096] \times [4096, 4096]$.
2. **GEMM (Prefill Simulation)**: Matrix-Matrix multiplication $[2048, 4096] \times [4096, 4096]$.


In [ ]:
# @title Day 4: Empirical CUDA Benchmarking Script
def run_cuda_kernel_benchmark(iterations: int = 100):
    """
    Empirically measures execution time, achieved TFLOPS, and memory bandwidth
    utilization for GEMV (decode) vs GEMM (prefill).
    """
    if not torch.cuda.is_available():
        print("⚠️ No CUDA GPU detected. Demonstrating benchmark methodology in simulation mode.")
        return
        
    device = torch.device("cuda:0")
    gpu_name = torch.cuda.get_device_name(0)
    
    # Dimensions: simulating Llama-3 hidden layer projection
    k, n = 4096, 4096
    
    # 1. Decode Simulation: GEMV (M=1)
    # 2. Prefill Simulation: GEMM (M=2048 tokens)
    scenarios = [
        {"type": "Decode (GEMV)", "m": 1},
        {"type": "Prefill (GEMM)", "m": 2048}
    ]
    
    # Allocate static weight matrix W [K, N] in FP16
    W = torch.randn(k, n, device=device, dtype=torch.float16)
    
    benchmark_results = []
    
    # CUDA Event timers
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)
    
    for sc in scenarios:
        m = sc["m"]
        X = torch.randn(m, k, device=device, dtype=torch.float16)
        
        # Warm-up iterations (to trigger CUDA initialization & autotuner)
        for _ in range(10):
            _ = torch.matmul(X, W)
        torch.cuda.synchronize()
        
        # Timed benchmark
        start_event.record()
        for _ in range(iterations):
            _ = torch.matmul(X, W)
        end_event.record()
        torch.cuda.synchronize()
        
        # Elapsed time in milliseconds
        elapsed_ms = start_event.elapsed_time(end_event) / iterations
        
        # Calculate FLOPs: 2 * M * K * N
        flops = 2.0 * m * k * n
        achieved_tflops = (flops / (elapsed_ms / 1000.0)) / 1e12
        
        # Memory traffic: Read X + Read W + Write Y
        bytes_transferred = (m * k * 2) + (k * n * 2) + (m * n * 2)
        achieved_bandwidth_gb_s = (bytes_transferred / (elapsed_ms / 1000.0)) / 1e9
        
        benchmark_results.append({
            "Scenario": sc["type"],
            "M (Batch/Tokens)": m,
            "Latency (ms)": round(elapsed_ms, 4),
            "Achieved TFLOPS": round(achieved_tflops, 2),
            "Achieved Bandwidth (GB/s)": round(achieved_bandwidth_gb_s, 2),
            "Operational Intensity": round(flops / bytes_transferred, 2)
        })
        
    df_bench = pd.DataFrame(benchmark_results)
    print(f"=== EMPIRICAL PROFILING RESULTS ON {gpu_name} ===")
    display(df_bench)

# Execute empirical benchmark
run_cuda_kernel_benchmark()


### 🔍 Deep Dive: What Did the Day 4 Empirical Test Reveal?

Notice the contrast in the empirical metrics:
* **Decode (GEMV, $M=1$)**: 
  - Achieved TFLOPS is practically near zero ($< 1$ to $5$ TFLOPS).
  - Achieved Bandwidth reaches close to the hardware's maximum physical memory bus limit!
  - Operational Intensity is $\approx 1.0 \text{ FLOP/Byte}$.
* **Prefill (GEMM, $M=2048$)**:
  - Achieved TFLOPS skyrockets into the tens or hundreds of TFLOPS (saturating Tensor Cores).
  - Operational Intensity jumps to $> 1000 \text{ FLOPs/Byte}$.


---
## 📊 Day 5: Synthesis: The Interactive Roofline Visualizer

Now we assemble the complete Roofline Diagram. We will plot:
1. The **theoretical ceilings** for NVIDIA H100 SXM5, A100 80GB, and your active accelerator.
2. The **operational trajectories** of token generation across batch sizes $B \in [1, 4, 16, 64, 256]$.
3. The exact transition boundary (Knee Point) where decode escapes memory starvation.


In [ ]:
# @title Day 5: Interactive Roofline Model Visualizer
def plot_master_roofline():
    """
    Generates a staff-grade Roofline Model chart visualizing hardware ceilings
    and operating points for prefill and decode regimes.
    """
    fig, ax = plt.subplots(figsize=(12, 7), dpi=150)
    
    # Operational Intensity range (log-scale from 0.1 to 10000 FLOPs/Byte)
    intensities = np.logspace(-1, 4, 500)
    
    # Target Accelerators to plot
    target_gpus = ["NVIDIA H100 SXM5 (80GB)", "NVIDIA A100 SXM4 (80GB)"]
    if torch.cuda.is_available():
        colab_gpu_name = torch.cuda.get_device_name(0)
        # Add active GPU if not already covered
        if not any(k in colab_gpu_name for k in ["H100", "A100"]):
            target_gpus.append("NVIDIA T4 (16GB - Colab Default)")
            
    colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]
    
    for idx, gpu_key in enumerate(target_gpus):
        spec = ACCELERATOR_DB[gpu_key]
        peak_tflops = spec["peak_fp16_tflops"]
        bw_tb_s = spec["peak_bandwidth_gb_s"] / 1000.0
        knee_pt = (peak_tflops / bw_tb_s)
        
        # Roofline equation: min(Peak Compute, Intensity * Bandwidth)
        attainable_perf = np.minimum(peak_tflops, intensities * bw_tb_s)
        
        ax.plot(intensities, attainable_perf, label=f"{gpu_key} (Ceiling: {peak_tflops} TF, Knee: {knee_pt:.0f} FLOP/B)",
                linewidth=2.5, color=colors[idx % len(colors)])
        
        # Mark Knee Point
        ax.scatter([knee_pt], [peak_tflops], color=colors[idx % len(colors)], s=60, zorder=5)
        ax.annotate(f"Knee: {knee_pt:.0f}", xy=(knee_pt, peak_tflops),
                    xytext=(knee_pt * 0.4, peak_tflops * 1.08),
                    fontweight='bold', color=colors[idx % len(colors)], fontsize=9)

    # Plot Operational Points for LLM Workloads
    workload_points = [
        {"name": "Decode (B=1)", "intensity": 1.0, "perf": 3.35, "color": "red", "marker": "o"},
        {"name": "Decode (B=8)", "intensity": 7.9, "perf": 26.5, "color": "darkorange", "marker": "s"},
        {"name": "Decode (B=32)", "intensity": 31.0, "perf": 103.8, "color": "gold", "marker": "^"},
        {"name": "Decode (B=128)", "intensity": 115.0, "perf": 385.0, "color": "lime", "marker": "p"},
        {"name": "Prefill (S=2048)", "intensity": 1024.0, "perf": 989.0, "color": "blue", "marker": "*"}
    ]
    
    for wp in workload_points:
        ax.scatter([wp["intensity"]], [wp["perf"]], color=wp["color"], s=100, zorder=6, edgecolors='black', label=wp["name"])
        ax.annotate(wp["name"], xy=(wp["intensity"], wp["perf"]),
                    xytext=(wp["intensity"] * 1.2, wp["perf"] * 0.8),
                    fontweight='bold', fontsize=9, bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="gray", alpha=0.8))

    # Architectural regions
    ax.axvspan(0.1, 100, alpha=0.08, color='red', label="Strictly Memory-Bound Regime")
    ax.axvspan(300, 10000, alpha=0.08, color='green', label="Compute-Bound Regime")
    
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim([0.1, 10000])
    ax.set_ylim([0.1, 2000])
    ax.set_xlabel("Operational Intensity (FLOPs / Byte) [Log Scale]", fontsize=12, fontweight='bold')
    ax.set_ylabel("Attainable Performance (TFLOPS) [Log Scale]", fontsize=12, fontweight='bold')
    ax.set_title("The LLM Serving Roofline Model: Why Decode is Memory-Bound", fontsize=14, fontweight='bold', pad=15)
    ax.legend(loc="lower right", frameon=True, fontsize=9)
    plt.tight_layout()
    plt.savefig("roofline_model_week1.png")
    plt.show()

plot_master_roofline()


### 🎯 Week 1 Staff Takeaways & Synthesis Checklist

1. **Decode is Memory-Bound**: At low batch sizes ($B=1$), the GPU memory bus is saturated transferring model weights, while Tensor Cores run at $< 1\%$ efficiency.
2. **Batching Rescues Operational Intensity**: As batch size increases ($B=1 \to B=128$), tokens in the batch share the weight matrix loaded into SRAM, walking the operational intensity up the slope toward the compute ceiling.
3. **Continuous Batching is Mandatory**: Static batching leaves GPUs idle whenever shorter sequences finish. Orca and vLLM continuous batching maintain constant operational intensity by inserting new requests at each token iteration.
4. **Quantization is Bandwidth Compression**: Moving from FP16 to FP8 or INT4 halves or quarters the bytes transferred across the memory bus, directly multiplying decode token throughput.

---

## 📝 Practice Exercises (Test Your Understanding)

### Exercise 1:
You are designing an inference cluster for a 70B parameter model in FP8 ($1 \text{ byte/param}$). Your marketing team promises users an interactive decode speed of $80 \text{ tokens/sec}$ per stream at $B=1$.
* **Question**: Can a single NVIDIA A100 80GB SXM ($2039 \text{ GB/s}$) physically deliver this SLO? What about an NVIDIA H100 SXM5 ($3350 \text{ GB/s}$)?
* *Hint*: Calculate $\frac{\text{Memory Bandwidth}}{\text{Model Weight Size}}$.

### Exercise 2:
Calculate the exact KV cache size in GB for a multi-tenant chat service running Llama-3-70B (GQA: 8 KV heads, $d_{\text{head}}=128$, $n_{\text{layers}}=80$) in FP16 for $256$ concurrent users with an average context length of $8192$ tokens. Does it fit on an 8-GPU H100 node?
